In [ ]:
**RDD 강의**: 이 노트북은 RDD 인터페이스(filter, map, reduceByKey) 학습용. Python worker 오류 시 HADOOP_HOME, Java 11/17, NumPy&lt;2 확인.

In [30]:
# Windows: 필수 설정 (커널 재시작 후 첫 셀 실행)
# NumPy 오류 시: 터미널에서 Batch/setup/fix_numpy.bat 실행 후 커널 재시작
import os
import sys

# 1) HADOOP_HOME (Parquet 읽기용)
for HADOOP_DIR in [
    r"E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5",
    r"E:\IT_SPACES\AI\ZoomCamp\DE\tools\hadoop-3.3.5",
]:
    if os.path.exists(os.path.join(HADOOP_DIR, "bin", "winutils.exe")):
        os.environ["HADOOP_HOME"] = HADOOP_DIR
        os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + os.path.join(HADOOP_DIR, "bin")
        print("HADOOP_HOME:", HADOOP_DIR)
        break
else:
    print("winutils 없음 → python Batch/setup/install_winutils.py")

# 2) Python worker 연결용 (RDD.take() 등에서 "Python worker failed to connect" 방지)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.python.worker.timeout", "120") \
    .getOrCreate()

HADOOP_HOME: E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5


In [31]:
# 1개월만 로드 (12개월 전체는 수천만 행이라 로딩 오래 걸림)
# 전체: 'data/pq/green/*/*'
df_green = spark.read.parquet('data/pq/green/2020/01')

```
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
```

In [32]:
# RDD 변환 
rdd = df_green.select('lpep_pickup_datetime', 'PULocationID', 'total_amount').rdd

In [33]:
from datetime import datetime

In [34]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [35]:
try:
    rows = rdd.take(10)
except Exception as e:
    print(f"[RDD 실패 → DataFrame 대안] {type(e).__name__}")
    rows = df_green.select('lpep_pickup_datetime', 'PULocationID', 'total_amount').filter("lpep_pickup_datetime >= '2020-01-01'").limit(10).collect()
row = rows[0]

[RDD 실패 → DataFrame 대안] Py4JJavaError


In [36]:
# RDD 실패 시 대안 (rdd.take(10) ≡ 아래와 동일)
# Connection reset / Python worker 오류 나면 이 셀 실행
rows = df_green.select('lpep_pickup_datetime', 'PULocationID', 'total_amount').filter("lpep_pickup_datetime >= '2020-01-01'").limit(10).collect()
row = rows[0]

In [37]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 18, 3, 52, 15), PULocationID=129, total_amount=6.3)

In [38]:
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [39]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [40]:
from collections import namedtuple

In [41]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [42]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [43]:
from pyspark.sql import types

RDD 파이프라인: filter → map → reduceByKey → map → toDF

In [44]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [45]:
# RDD 연산 (강의 핵심: filter, map, reduceByKey, unwrap)
try:
    df_result = rdd \
        .filter(filter_outliers) \
        .map(prepare_for_grouping) \
        .reduceByKey(calculate_revenue) \
        .map(unwrap) \
        .toDF(result_schema)
    df_result.write.mode("overwrite").parquet('tmp/green-revenue')
except Exception as e:
    print(f"[RDD 실패 → DataFrame 대안] {type(e).__name__}")
    from pyspark.sql import functions as F
    df_result = df_green.filter(F.col("lpep_pickup_datetime") >= "2020-01-01") \
        .withColumn("hour", F.date_trunc("hour", F.col("lpep_pickup_datetime"))) \
        .groupBy("hour", "PULocationID").agg(F.sum("total_amount").alias("revenue"), F.count(F.lit(1)).alias("count")) \
        .withColumnRenamed("PULocationID", "zone")
    df_result.write.mode("overwrite").parquet('tmp/green-revenue')

[RDD 실패 → DataFrame 대안] Py4JJavaError


In [46]:
# RDD 실패 시 대안 (동일 결과: filter→groupBy)
# Connection reset / Python worker 오류 나면 이 셀 실행
from pyspark.sql import functions as F
df_result = df_green.filter(F.col("lpep_pickup_datetime") >= "2020-01-01") \
    .withColumn("hour", F.date_trunc("hour", F.col("lpep_pickup_datetime"))) \
    .groupBy("hour", "PULocationID").agg(F.sum("total_amount").alias("revenue"), F.count(F.lit(1)).alias("count")) \
    .withColumnRenamed("PULocationID", "zone")
df_result.write.mode("overwrite").parquet('tmp/green-revenue')

In [47]:
df_result.show(5)

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-17 20:00:00|  41| 633.6500000000001|   66|
|2020-01-03 18:00:00| 223|313.65999999999997|   25|
|2020-01-22 07:00:00|  33|            185.72|   10|
|2020-01-26 20:00:00|  89|46.879999999999995|    2|
|2020-01-05 10:00:00| 244|241.14000000000001|   15|
+-------------------+----+------------------+-----+
only showing top 5 rows



In [48]:
# mapPartitions 예제 (배치 단위 처리)

In [49]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green.select(columns).rdd

In [50]:
import pandas as pd

In [51]:
try:
    rows = duration_rdd.take(10)
except Exception:
    rows = df_green.select(columns).limit(10).collect()

In [52]:
# RDD 실패 시 대안: duration_rdd.take(10) ≡ 아래와 동일
# rows = duration_rdd.take(10)
rows = df_green.select(columns).limit(10).collect()

In [53]:
df = pd.DataFrame(rows, columns=columns)

In [54]:
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [55]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [56]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [57]:
# mapPartitions: 파티션 단위로 pandas 배치 처리
try:
    df_predicts = duration_rdd \
        .mapPartitions(apply_model_in_batch) \
        .toDF() \
        .drop('Index')
except Exception as e:
    print(f"[mapPartitions 실패 → DataFrame 대안] {type(e).__name__}")
    from pyspark.sql import functions as F
    df_predicts = df_green.select(columns).withColumn("predicted_duration", F.col("trip_distance") * 5)

[mapPartitions 실패 → DataFrame 대안] Py4JJavaError


In [58]:
# RDD 실패 시 대안: mapPartitions ≡ trip_distance * 5
# df_predicts = duration_rdd.mapPartitions(...)  # 위가 실패하면 아래 실행
from pyspark.sql import functions as F
df_predicts = df_green.select(columns).withColumn("predicted_duration", F.col("trip_distance") * 5)

In [59]:
df_predicts.select('predicted_duration').show()

+------------------+
|predicted_duration|
+------------------+
|3.4499999999999997|
|             16.95|
|              4.55|
|              14.0|
|             35.45|
|              15.5|
| 8.100000000000001|
|               6.5|
| 8.299999999999999|
|               0.0|
|              4.65|
|              2.05|
|               8.0|
|              70.8|
|             49.85|
|             61.25|
|              6.05|
| 93.69999999999999|
|             72.95|
|             12.15|
+------------------+
only showing top 20 rows

